## Découverte de LoRA : Low-Rank Adaptation of Large Language Models

### Le Fine-Tuning
L'approche classique pour spécialiser un modèle pré-entraîné à une tâche particulière est le **fine-tuning**. Il s'agit simplement de partir des poids du pré-entraînement et de poursuivre l'entraînement sur des données spécifiques à la tâche demandée. 

> ⚠️ **Attention :** il faut ajuster le learning rate en conséquence pour ne pas perdre la qualité du pré-entraînement et uniquement ajuster les poids. 

Avec cette approche, on peut aussi **geler** des couches du modèle pour, par exemple, *fine-tuner* uniquement la tête du modèle et pas toute l'architecture.

### La méthode LoRA et ses avantages
Le papier [LoRA : Low-Rank Adaptation of Large Language Models](https://arxiv.org/abs/2106.09685) met en avant une nouvelle approche pour spécialiser un modèle en utilisant moins de paramètres. Ce qui permet d'avoir plusieurs avantages : 

* **Réduction du stockage global** : on peut utiliser un même modèle pré-entraîné pour plusieurs tâches différentes, on a juste à avoir des modules d'ajustement LoRA pour chacune ;
* **Entraînement plus efficace** : plus rapide et moins coûteux, on calcule uniquement le gradient de matrices de basses dimensions ;
* **Aucune latence sur l'inférence** : grâce à une architecture linéaire qui nécessite simplement de sommer les poids pré-entraînés aux poids adaptés ;
* **Combinaison possible** : LoRA peut être combiné à des méthodes de pre-tuning comme le *prefix-tuning*.



Concrètement, au lieu de modifier nos poids pré-entraînés $W_0\in\mathbb{R}^{d\times k}$, on va geler $W_0$ et entraîner une matrice $\Delta W\in\mathbb{R}^{d\times k}$ que l'on vient sommer à $W_0$. Soit une couche $h$ et une entrée $x$, on a : 
$$
h = W_0 x + \Delta W x
$$
Pour des tâches différentes, on peut donc avoir plusieurs $\Delta W, \Delta W', ...$ pour chacune des tâches et donc simplement remplacer le $\Delta W$ par la matrice correspondante à la tâche souhaitée.
Jusqu'ici on a toujours $d\times k$ paramètres à entraîner. Pour réduire le nombre de paramètres, **LoRA** vient décomposer cette matrice $\Delta W$ en $BA$ avec $B\in\mathbb{R}^{d\times r}$ et $A\in\mathbb{R}^{r\times k}$ avec $r\ll\min(d,k)$ le rang. On a ainsi $d\times r + r\times k$ paramètres à entraîner. Soit une couche $h$ et une entrée $x$, on a : 
$$
h = W_0 x + \Delta W x = W_0 x + BA x  
$$

On utilise cette reparametrisation pour réduire le nombre de paramètre à entrainer et pour garder les poids pré-entraînés tels quels. On a seulement à entraîner les matrices de basses dimensions $A$ et $B$ (voir l'illustration : Figure 1 du papier). 

![LoRA reparametrization. We only train $A$ and $B$](https://arxiv.org/html/2106.09685v2/x1.png)

Pour l'initialisation, on utilise une initialisation Gaussienne $N(0,\sigma^2)$ pour $A$ et une initialisation à zéro pour $B$ de tel sorte que $\Delta W = BA$ est initialisé à zéro au début de l'entraînement. Durant ce dernier, on met à l'échelle $\Delta Wx$ en multipliant par $\dfrac{\alpha}{r}$ avec $\alpha$ une constante en $r$ (souvent $\alpha=r$ ou $\alpha=2r$), cette constant peut être vue comme le learning rate de l'entraînement.


Faits notables dans les exemples sur le papier : 
- La méthode LoRA outperform toutes les autres sur le E2E NLG challenge ;
- On passe de 354.92M de paramètre sur GPT-2 medium à 0.35M de paramètres soit un facteur $÷1000$ ;
- On gagne 25% de temps d'entraînement sur GPT-3 157B face à un fine-tuning complet ;
- Les paramètres entraînables peuvent être jusqu'à $0.01$% du nombre de paramètres de $W_0$.

### LoRA from scratch 

Il est maintenant temps d'implémenter cette méthode en partant de rien, puis nous verrons comment s'en servir en application.




In [21]:
import torch
import torch.nn as nn

class LoraLayer(nn.Module):
    def __init__(self, in_dim, out_dim, rank, alpha):
        super().__init__()
        self.w_orig = nn.Linear(in_dim, out_dim) # Poids gelés
        self.w_orig.weight.requires_grad = False
        
        # Matrices LoRA
        self.A = nn.Parameter(torch.randn(rank, in_dim))
        self.B = nn.Parameter(torch.zeros(out_dim, rank))
        self.scaling = alpha / rank

    def forward(self, x):
        # Chemin classique + chemin LoRA
        return self.w_orig(x) + (x @ self.A.T @ self.B.T) * self.scaling

In [22]:
def inject_lora(model, rank, alpha):
    for name, module in model.named_children():
        if isinstance(module, nn.Linear):
            # On remplace la couche linéaire par notre version LoRA
            old_layer = module
            new_layer = LoraLayer(old_layer.in_features, old_layer.out_features, rank, alpha)
            # On copie les poids originaux gelés dans la nouvelle couche
            new_layer.w_orig.weight.data = old_layer.weight.data.copy_()
            setattr(model, name, new_layer)
        else:
            # Récursion pour descendre dans les sous-modules (ex: Attention blocks)
            inject_lora(module, rank, alpha)

In [23]:
from transformers import AutoModelForCausalLM

In [24]:
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
